# 2.3 Grounding Language Models in Scientific Literature

In the previous section, we asked a language model a scientific question without providing any external sources. The model produced a plausible answer, but its claims were based only on information learned during training.

We can improve this workflow by supplying relevant scientific literature directly to the model. We will begin with a single abstract, then expand the same approach to retrieve and use multiple papers.

## 2.3.1 Retrieving a Scientific Abstract from PubMed

In the previous section, the language model answered our question using only information learned during training. We can instead provide the model with evidence from the scientific literature.

PubMed provides an API through [NCBI's E-utilities](https://www.ncbi.nlm.nih.gov/books/NBK25500/) that allows us to search for papers and retrieve article information directly from Python. We'll first search PubMed for a paper relevant to our question, then pass its abstract to the same local language model.

In [45]:
import requests
import xml.etree.ElementTree as ET

query = (
    '("kinase inhibitors"[Title]) AND '
    '(cancer[Title]) AND '
    '(limitations[Title/Abstract]) AND '
    'review[Publication Type]'
)

search_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"

params = {
    "db": "pubmed",
    "term": query,
    "retmax": 5,
    "retmode": "xml",
}

response = requests.get(search_url, params=params)
root = ET.fromstring(response.content)

pubmed_ids = [
    element.text
    for element in root.findall(".//IdList/Id")
]

print("PMIDs:", ", ".join(pubmed_ids))

PMIDs: 42539807, 41642503, 41584508, 40921183, 40831326


PubMed returns a unique identifier for each matching article. These PMIDs can be used to retreive information about the paper like their titles, authors, and abstracts.

The search above gives us candidate papers, but a PMID along doesn't tell use whether a paper contains the evidence we need. Next, we'll retreive the article records themselves.

In [46]:
fetch_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"

params = {
    "db": "pubmed",
    "id": ",".join(pubmed_ids),
    "retmode": "xml",
}

response = requests.get(fetch_url, params=params)
root = ET.fromstring(response.content)

papers = []

for article in root.findall(".//PubmedArticle"):
    title = article.findtext(".//ArticleTitle")
    
    abstract_parts = article.findall(".//AbstractText")
    abstract = " ".join(
        part.text for part in abstract_parts
        if part.text
    )
    
    pmid = article.findtext(".//PMID")

    papers.append({
        "pmid": pmid,
        "title": title,
        "abstract": abstract,
    })

for paper in papers:
    print(f"PMID: {paper['pmid']}")
    print(f"Title: {paper['title']}")
    print()

PMID: 42539807
Title: Research progress of small molecule protein kinase inhibitors (SMKIs) in the treatment of colorectal cancer: mechanism, application, and future prospects.

PMID: 41642503
Title: Therapeutic Potential of Tyrosine Kinase Inhibitors in Advanced Thyroid Cancer.

PMID: 41584508
Title: Recent FDA-approved kinase inhibitors for cancer therapy in 2025: A comprehensive review and perspectives.

PMID: 40921183
Title: Repurposing FDA-Approved Drugs as Fructosamine-3-Kinase Inhibitors: A Mechanistic and Translational Approach to Redox-Driven Cancer Therapy.

PMID: 40831326
Title: Progress in Nanocarriers-Based Approaches for the Delivery of Tyrosine Kinase Inhibitors in Bone Cancer: Trends and Prospects.



PubMed returns papers that match the search terms, but the first results may not always be the best sources for the question you want to answer. In practice, literature retrieval often requires some query refinement. You may need to add or remove keywords, use PubMed field tags, or restrict the search to particular article types.

In [47]:
selected_paper = papers[2]

print(f"PMID: {selected_paper['pmid']}")
print(f"Title: {selected_paper['title']}")
print(f"Abstract: {selected_paper['abstract']}")

PMID: 41584508
Title: Recent FDA-approved kinase inhibitors for cancer therapy in 2025: A comprehensive review and perspectives.
Abstract: Malignant disorders continue to represent one of the major burdens of disease globally, especially in the context of premature deaths. Targeted anticancer treatments, including kinase inhibitors (KIs), have become crucial tools to disrupt the specific signaling pathways that are responsible for cancer growth following malignant transformation. Evidence demonstrates that KIs have substantially advanced precision oncology across multiple malignancies, with clinical success most notable in hematologic cancers and specific solid tumors, such as non-small cell lung cancer. Nonetheless, their long-term efficacy is often constrained by the emergence of acquired resistance, intratumoral heterogeneity, and off-target toxicities, underscoring the need for adaptive therapeutic strategies and combination regimens. While next-generation KIs and ongoing trials of

### Answering with Scientific Context

We'll use the same local language model and chat format introduced in Section 2.2, but with a different prompt. 

Previously, the model received only the scientific question. This time we will include the title and abstract of the selected PubMed paper in the prompt and instruct the model to base its answer on that source.

This allows us to ask the same question again while giving the model relevant scientific evidence to use in its response.

In [50]:
!pip install -q transformers accelerate ipywidgets requests


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [51]:
from transformers import pipeline

model = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-1.5B-Instruct",
    device_map="auto"
)

Device set to use mps


In [70]:
question = (
    "How do kinase inhibitors work in cancer treatment, "
    "and what are some major limitations of this therapeutic approach?"
)

messages = [
    {
        "role": "system",
        "content": (
            "You are a scientific assistant. "
            "Answer the question using only the scientific source provided. "
            "If the source does not contain enough information to answer part of the question, say so."
        )
    },
    {
        "role": "user",
        "content": f"Scientific source:\nTitle: {selected_paper['title']}\nAbstract: {selected_paper['abstract']}\n Question: {question}"
    }
]

prompt = model.tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

response = model(
    prompt,
    max_new_tokens=1024,
    do_sample=False,
    temperature=None,
    top_p=None,
    top_k=None,
)

print(response[0]["generated_text"][len(prompt):])

Kinase inhibitors target proteins called kinases, which play key roles in regulating cellular processes like proliferation, differentiation, and apoptosis. By inhibiting these enzymes, kinase inhibitors can block signals that promote tumor growth or spread.

Some major limitations of kinase inhibitor use include:

1. Acquired resistance: Tumors may develop mutations that allow them to bypass the effects of the inhibitor, rendering it ineffective over time.

2. Intratumoral heterogeneity: Different parts of a tumor might respond differently to the same kinase inhibitor, limiting its effectiveness overall.

3. Off-target toxicities: These drugs can affect normal cells, causing side effects unrelated to cancer treatment.

4. Uneven distribution of clinical benefits: Some cancers benefit more from kinase inhibitors than others, highlighting gaps in treatment efficacy.

5. Disparities in access and affordability: Limited availability and high costs make these therapies less accessible to ma

The second response is more closely tied to the scientific literature because the model was given a relevant abstract as part of its prompt.

However, a single abstract provides only a limited view of the literature. Broader scientific questions require evidence from multiple studies, but manually finding and pasting each relevant paper quickly becomes impractical.

This is the motivation behind **retrieval augmented generation (RAG)**. Instead of selecting one source by hand, a RAG workflow retrieves a set of relevant documents for a question and supplies that evidence to the language model before it generates an answer.

In the next step, we will extend the PubMed workflow to retrieve multiple abstracts, identify the most relevant results, and use them together as context for the same question.

## 2.3.2 Retrieving Multiple Papers

Instead of selecting one paper at a time, we can extend our PubMed search to retrieve a larger set of potentially relevant abstracts. We can then identify which of those papers are most useful for answering our question.

This separates the workflow into two steps:

1. Retrieval: collect candidate papers from PubMed.
2. Ranking: determine which retrieved papers are most relevant to the question.

We will start by retrieving a larger set of abstracts using the same PubMed API workflow introduced above.

In [60]:
def search_pubmed(query, max_results=20):
    # Search PubMed for matching article IDs
    search_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"

    search_params = {
        "db": "pubmed",
        "term": query,
        "retmax": max_results,
        "retmode": "xml",
    }

    response = requests.get(search_url, params=search_params)
    root = ET.fromstring(response.content)

    pubmed_ids = [
        element.text
        for element in root.findall(".//IdList/Id")
    ]

    # Fetch article records for those IDs
    fetch_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"

    fetch_params = {
        "db": "pubmed",
        "id": ",".join(pubmed_ids),
        "retmode": "xml",
    }

    response = requests.get(fetch_url, params=fetch_params)
    root = ET.fromstring(response.content)

    papers = []

    for article in root.findall(".//PubmedArticle"):
        pmid = article.findtext(".//PMID")
        title = article.findtext(".//ArticleTitle")

        abstract_parts = article.findall(".//AbstractText")
        abstract = " ".join(
            part.text for part in abstract_parts
            if part.text
        )

        papers.append({
            "pmid": pmid,
            "title": title,
            "abstract": abstract,
        })

    return papers

In [61]:
papers = search_pubmed(
    query=query,
    max_results=20
)

print(f"Retrieved {len(papers)} papers")

Retrieved 20 papers


### Ranking Papers by Relevance

A PubMed search can return many papers that match the query, but not every result will be equally useful for answering our specific question.

To narrow the candidate set, we can rank the retrieved abstracts according to how closely they match the question.

Here, we will use **BM25**, a widely used information-retrieval algorithm that scores documents based on how well their contents match the words in a query.

In [64]:
!pip install -q rank_bm25 nltk

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [67]:
import nltk
from rank_bm25 import BM25Okapi

nltk.download("punkt_tab")

tokenized_abstracts = [
    nltk.word_tokenize(paper["abstract"].lower())
    for paper in papers
]

bm25 = BM25Okapi(tokenized_abstracts)

tokenized_question = nltk.word_tokenize(question.lower())

scores = bm25.get_scores(tokenized_question)

top_papers = [
    papers[i]
    for i in sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:5]
]

for paper in top_papers:
    print(f"PMID: {paper['pmid']}")
    print(f"Title: {paper['title']}")
    print()

PMID: 36183569
Title: Lost in translation: Revisiting the use of tyrosine kinase inhibitors in colorectal cancer.

PMID: 32690442
Title: Kinase inhibitors with viral oncolysis: Unmasking pharmacoviral approaches for cancer therapy.

PMID: 27038473
Title: Multi-kinase inhibitors, AURKs and cancer.

PMID: 28669349
Title: Recent Updates on the Therapeutic Potential of HER2 Tyrosine Kinase Inhibitors for the Treatment of Breast Cancer.

PMID: 41642503
Title: Therapeutic Potential of Tyrosine Kinase Inhibitors in Advanced Thyroid Cancer.



[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/benjaminsiciliano/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


A single abstract many only address part of the problem, while the highest-ranked papers may provide broader coverage because each contributes different pieces of information relevant to the question. 

We can now combine the retrieved abstracts and provide them to the language model as context. This patten of retrieving relevant information before generating an answer is the basis of RAG.

We can now pass those soruces to the same local language model used earlier. The model itself has not changed. The difference is that the prompt now contains multiple retrieved sources rather than a single manually selected abstract.

In [71]:
context = "\n\n".join(
    f"Title: {paper['title']}\nAbstract: {paper['abstract']}"
    for paper in top_papers
)

messages = [
    {
        "role": "system",
        "content": (
            "You are a scientific assistant. "
            "Answer the question using only the scientific sources provided. "
            "If the sources do not contain enough information to answer part "
            "of the question, say so."
        )
    },
    {
        "role": "user",
        "content": f"Scientific sources: {context}\nQuestion: {question}"
    }
]

prompt = model.tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

response = model(
    prompt,
    max_new_tokens=1024,
    do_sample=False,
    temperature=None,
    top_p=None,
    top_k=None,
)

print(response[0]["generated_text"][len(prompt):])

Kinase inhibitors work by blocking enzymes called kinases, which are crucial for cell signaling processes. When these enzymes are inhibited, they cannot activate downstream molecules, leading to reduced signal transduction and ultimately cell death. This mechanism allows kinase inhibitors to selectively target cancer cells while sparing normal cells, making them potential treatments for various types of cancer.

However, there are several major limitations to the use of kinase inhibitors in cancer treatment:

1. **Cytostatic Effect**: Many kinase inhibitors act as cytostatics, meaning they prevent cells from dividing but do not kill them outright. This can lead to prolonged periods of dormancy before the cancer cells eventually die due to exhaustion of energy stores.

2. **Resistance Mechanisms**: Cancer cells often develop resistance to kinase inhibitors through genetic mutations, epigenetic changes, or alterations in the expression of other genes that compensate for the loss of kinas

The model now has access to several retrieved abstracts rather than a single source, allowing it to draw on a broader set of information when answering the question.

However, retrieval alone does not guarantee that every statement in the generated answer is supported by the retrieved evidence. The language model still generates text probabilistically and may introduce claims that are not clearly present in the supplied sources.

A basic RAG workflow therefore has two important components: retrieving relevant evidence and instructing the model to use that evidence carefully. More advanced systems can add additional steps for source selection, citation, verification, and iterative retrieval.

### Putting the Workflow Together

We have now built a simple literaure-based RAG workflow:

1. Search PubMed for candidate papers.
2. Retrieve their abstracts.
3. Rank the abstracts by relevance to the question.
4. Provide the highest-ranked sources to a language model.
5. Generate an answer using the retrieved evidence.

This same general pattern can be applied to other scientific data sources. 

In the next sections, we will use language models with structured biological databases rather than scientific literature.

### Further Reading

The RAG workflow in this tutorial is intentionally simple. Scientific systems can extend this pattern with full-text retrieval, citation tracking, iterative search, evidence scoring, and verification.

One example is [PaperQA2](https://www.futurehouse.org/news/wikicrow), a scientific RAG system developed by FutureHouse for answering research questions from the literature. It is designed to retrieve, evaluate, and synthesize evidence across scientific papers rather than relying on a single retrieval step.

For readers interested in more advanced scientific literature workflows, see the PaperQA2 [paper](https://arxiv.org/abs/2409.13740) and [documentation](https://github.com/future-house/paper-qa).